# 📈 03 — Statistical Spillover Baseline (Diebold–Yilmaz 2012)

Corrected replacement for the archived `02_statistical_spillover_baseline.ipynb` (see `notebooks/archive/README.md`).

1. **Framework:** Diebold–Yilmaz (2012) spillover index with the order-invariant **generalized forecast error variance
   decomposition (GFEVD)** of Pesaran & Shin (1998).
2. **Total Spillover Index (TSI):** the share of forecast-error variance that comes from other coins.
3. **Directional table:** how much each coin receives FROM and sends TO the others; **NET** = TO − FROM.
4. **Export:** `data/processed/statistical_connectedness.csv`, the ranking the TempME explanations are validated against.

The VAR needs a balanced panel, so it is fitted on **Panel A**: the 20 largest coins with complete data from 2017-11-09,
using **raw daily log returns** from the cleaned prices.

In [1]:
import sys
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from cryptograph.analysis import VARSpilloverBaseline

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"

panel = torch.load("../data/processed/v1/panel.pt", weights_only=False)
close = pd.read_parquet("../data/interim/master.parquet")["close"].unstack("coin")
panel_a = [c for c, a in zip(panel["coins"], panel["panel_A"]) if a]
common_returns = np.log(close[panel_a]).diff().loc["2017-11-09":].dropna()
print(f"📊 Panel A: {len(panel_a)} coins, {len(common_returns)} days "
      f"({common_returns.index[0].date()} → {common_returns.index[-1].date()})")
print("🪙", ", ".join(panel_a))

📊 Panel A: 20 coins, 3244 days (2017-11-10 → 2026-09-27)
🪙 BTC, ETH, BNB, XRP, TRX, ZEC, DOGE, LINK, XMR, ADA, XLM, BCH, LTC, ETC, DASH, XTZ, DCR, NEO, DGB, LSK


*What this shows:* Panel A is balanced: all 20 coins have a price on every day from 2017-11-09, so the VAR needs no filling. Some of these
coins (DCR, DGB, LSK, …) are small today; they are in Panel A because they have the full history the VAR requires.

## 1. Fit the VAR and compute the generalized FEVD

$$r_t = c + \sum_{l=1}^{p} \Phi_l\, r_{t-l} + \varepsilon_t, \qquad
\theta_{ij}(H) = \frac{\sigma_{jj}^{-1}\sum_{h=0}^{H-1}(e_i' A_h \Sigma e_j)^2}{\sum_{h=0}^{H-1} e_i' A_h \Sigma A_h' e_i}$$

$\theta_{ij}(H)$ is the share of coin $i$'s $H$-day forecast uncertainty explained by shocks to coin $j$ (rows normalised to 100%).

In [2]:
baseline = VARSpilloverBaseline(maxlags=2, forecast_horizon=10)
baseline.fit(common_returns)
spillover_results = baseline.compute_spillover_table()          # method="generalized" (default)
tsi = spillover_results["total_spillover_index"]
print(f"📊 Total Spillover Index (TSI): {tsi:.2f}%  (VAR lag order {baseline.fitted_model.k_ar}, horizon 10 days)")
print(f"   {tsi:.1f}% of each coin's 10-day forecast-error variance comes from shocks in other coins.")

[2026-09-30 22:37:28] [INFO] [cryptograph.analysis.var_spillover] - Fitted VAR baseline with lag order: 2


📊 Total Spillover Index (TSI): 86.03%  (VAR lag order 2, horizon 10 days)
   86.0% of each coin's 10-day forecast-error variance comes from shocks in other coins.


C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


*What this shows:* A Total Spillover Index of about 86% means the crypto market is highly interconnected: most of each coin's future
uncertainty comes from the others, not from itself. (The archived notebook's 54% came from the Cholesky method, whose
result depends on the order of the columns.)

## 2. Directional spillover table (FROM others vs TO others)

In [3]:
table = spillover_results["spillover_matrix"].copy()
table.loc["TO"] = list(spillover_results["to_others"].values) + [np.nan]
display(table.round(1))

,BTC,ETH,BNB,XRP,TRX,ZEC,DOGE,LINK,XMR,ADA,...,BCH,LTC,ETC,DASH,XTZ,DCR,NEO,DGB,LSK,FROM
BTC,11.5,7.4,5.2,3.9,3.4,4.1,3.4,4.2,5.1,4.8,...,5.5,6.3,5.1,4.3,4.1,4.2,5.4,4.4,3.5,88.5
ETH,6.8,10.5,4.7,4.4,3.2,4.3,2.9,4.9,4.5,5.1,...,5.5,6.4,6.0,4.6,4.2,3.9,5.8,4.2,3.7,89.5
BNB,6.5,6.5,14.6,3.7,3.4,4.0,2.4,4.4,4.7,4.5,...,4.5,5.7,4.8,4.2,4.3,3.6,5.6,4.4,3.9,85.4
XRP,4.8,5.9,3.5,14.1,3.7,4.2,3.0,4.1,3.7,6.3,...,4.6,5.7,5.2,4.0,3.9,2.9,5.3,3.9,3.3,85.9
TRX,5.5,5.6,4.1,4.7,17.9,3.4,2.6,3.8,4.7,5.0,...,4.6,5.1,4.8,4.0,3.7,3.1,5.4,4.4,3.5,82.1
ZEC,4.9,5.7,3.8,4.1,2.7,13.9,2.4,4.1,5.3,4.4,...,5.1,5.6,5.4,7.5,3.8,3.9,5.5,4.0,3.6,86.1
DOGE,5.7,5.4,3.2,4.1,2.8,3.5,19.5,3.8,3.5,4.8,...,4.7,5.2,5.7,3.9,3.3,3.3,5.0,4.6,3.2,80.5
LINK,5.2,6.7,4.4,4.2,3.1,4.2,2.8,14.3,3.8,5.2,...,4.6,5.3,5.2,4.2,4.9,3.6,5.5,4.4,3.8,85.7
XMR,6.2,6.0,4.5,3.7,3.7,5.3,2.5,3.8,14.2,4.4,...,5.2,5.6,4.9,5.4,3.7,3.8,5.4,4.1,3.7,85.8
ADA,5.2,6.2,3.8,5.7,3.4,4.0,3.1,4.6,4.0,12.8,...,4.6,5.4,5.6,3.9,4.2,3.2,5.5,4.6,3.3,87.2


*What this shows:* Row *i*, column *j* is the percentage of coin *i*'s forecast-error variance caused by coin *j*; the diagonal is the coin's
own share (10.5–19.5%). The **FROM** column sums each row's off-diagonal entries (what a coin receives), and the **TO**
row sums each column (what a coin sends). Unlike the archived table, no single coin dominates every row, because the
generalized FEVD does not depend on which coin is listed first.

## 3. NET connectedness ranking (transmitters vs receivers)

In [4]:
net_spillover = spillover_results["net_spillover"].sort_values(ascending=False)
net_df = pd.DataFrame({
    "Asset": net_spillover.index, "Net_Spillover": net_spillover.values,
    "TO": spillover_results["to_others"][net_spillover.index].values,
    "FROM": spillover_results["from_others"][net_spillover.index].values,
    "Role": ["Net Transmitter" if v > 0 else "Net Receiver" for v in net_spillover.values]})
fig = px.bar(net_df, x="Asset", y="Net_Spillover", color="Role",
             color_discrete_map={"Net Transmitter": "#00e676", "Net Receiver": "#ff1744"},
             title=f"Diebold–Yilmaz NET spillover ranking (TSI = {tsi:.2f}%)",
             labels={"Net_Spillover": "NET spillover (percentage points)", "Asset": "Cryptocurrency"})
fig.update_layout(height=450, xaxis_tickangle=-45)
fig.show()

transmitters = net_spillover[net_spillover > 0]
print("Top 3 net transmitters:")
for a, v in transmitters.head(3).items():
    print(f"  ▲ {a:5s} {v:+6.2f} pp")
print("Top 3 net receivers:")
for a, v in net_spillover.tail(3)[::-1].items():
    print(f"  ▼ {a:5s} {v:+6.2f} pp")

Top 3 net transmitters:
  ▲ ETH   +27.80 pp
  ▲ LTC   +18.86 pp
  ▲ NEO   +17.33 pp
Top 3 net receivers:
  ▼ DOGE  -26.07 pp
  ▼ TRX   -21.25 pp
  ▼ DCR   -15.17 pp


*What this shows:* Green bars send more shock variance to the others than they receive (ETH, LTC, NEO, BTC lead); red bars mostly absorb
shocks (DOGE, TRX, DCR, LSK). Values are percentage points of variance (TO − FROM), not percent returns.

## 4. Export the benchmark for TempME validation

In [5]:
out_path = Path("../data/processed/statistical_connectedness.csv")
net_df.to_csv(out_path, index=False)
print(f"💾 Saved {len(net_df)} coins (Asset, Net_Spillover, TO, FROM, Role) to {out_path}")

💾 Saved 20 coins (Asset, Net_Spillover, TO, FROM, Role) to ..\data\processed\statistical_connectedness.csv


*What this shows:* This file is the statistical ground truth. Later, TempME's coin-importance scores from the TGN are compared with this NET
ranking (Spearman rank correlation) to check that the model's view of who spreads shocks agrees with the statistics.